# Documentation

This code calculates the normalized gross moist stability (nGMS) following Inoue & Back (2015).

# Imports

In [5]:
%load_ext autoreload
%autoreload 2

print("Loading imports...")
from config import *

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Data analysis
import glob
import os
import sys
from datetime import datetime, timedelta
import copy
import cftime

# Plotting
import matplotlib.pyplot as plt
import numpy as np
import scipy
import scipy.signal as signal
from scipy.stats import t
import xarray as xr
import xrft

# Cartopy
from cartopy import crs as ccrs
from cartopy import feature as cf
from cartopy import util as cutil
from matplotlib import colors as mcolors
from matplotlib import ticker as mticker
from matplotlib.gridspec import GridSpec
from scipy.optimize import curve_fit

# import colormaps as cmaps
sys.path.insert(0, "/glade/u/home/sressel/auxiliary_functions/")
import mjo_mean_state_diagnostics as mjo
from bmh_colors import bmh_colors
from one_two_one_filter import one_two_one_filter
from rounding_functions import round_out
from tick_labeller import tick_labeller
from standardize_data import standardize_data

from processing_functions import time_filter_data, mjo_filter_data
from load_aquaplanet_data import *

print("Imports loaded")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loading imports...
Imports loaded


# Specify parameters

In [14]:
# Set latitude bounds
LATITUDE_SOUTH = -30
LATITUDE_NORTH = 30
latitude_subset_bounds = slice(LATITUDE_SOUTH, LATITUDE_NORTH)

# Set central longitude
CENTRAL_LONGITUDE = 0

# Set longitude bounds
LONGITUDE_MIN = 0
LONGITUDE_MAX = 360

# Set time bounds
missing_days = [
    cftime.DatetimeNoLeap(7, 2, 5, 0, 0, 0, 0, has_year_zero=True),
    cftime.DatetimeNoLeap(7, 2, 6, 0, 0, 0, 0, has_year_zero=True),
    cftime.DatetimeNoLeap(7, 2, 7, 0, 0, 0, 0, has_year_zero=True),
]

START_TIME = cftime.DatetimeNoLeap(3, 1, 3, 0, 0, 0, 0, has_year_zero=True)
# END_TIME = cftime.DatetimeNoLeap(13, 1, 3, 0, 0, 0, 0, has_year_zero=True)
END_TIME = cftime.DatetimeNoLeap(3, 1, 8, 0, 0, 0, 0, has_year_zero=True)
first_half_subset_bounds = slice(START_TIME, missing_days[0] - timedelta(days=1))
second_half_subset_bounds = slice(missing_days[-1] + timedelta(days=1), END_TIME)

# Cut-off periods for intraseasonal filtering
INTRASEASONAL_LOWCUT = 100
INTRASEASONAL_HIGHCUT = 20
frequency_subset_bounds = slice(INTRASEASONAL_LOWCUT, INTRASEASONAL_HIGHCUT)

# Cut-off wavenumbers for MJO-filtering
LARGE_SCALE_CUTOFF = 1
SMALL_SCALE_CUTOFF = 3
wavenumber_bounds = slice(LARGE_SCALE_CUTOFF, SMALL_SCALE_CUTOFF)

# Column-integrated bounds
LOWER_LEVEL_PRESSURE = 950
UPPER_LEVEL_PRESSURE = 100
pressure_subset_bounds = slice(UPPER_LEVEL_PRESSURE, LOWER_LEVEL_PRESSURE)

In [15]:
def beta_column_integrate(pressure_array, variable, surface_pressure, pbot, ptop):
    """
    Perform vertical integration of a variable using Trenberth (1991) beta factors.

    This function computes the vertical integral of an atmospheric variable
    using pressure levels and surface pressure, accounting for partial layer
    contributions via the beta factor method described by Trenberth (1991).

    Parameters
    ----------
    pressure_array : xarray.DataArray
        A 4D array of pressure values with dimensions (time, lat, lon, index),
        where `index` refers to the midpoint of pressure layers (odd-indexed).

    variable : xarray.DataArray
        A 4D array of the variable to be integrated, with dimensions
        (time, lat, lon, lev), where `lev` matches the vertical structure
        of `pressure_array` (e.g., pressure levels).

    surface_pressure : xarray.DataArray
        A 3D array (time, lat, lon) of surface pressure values.

    pbot : float
        Bottom pressure level (e.g., 100000 Pa for surface).

    ptop : float
        Top pressure level for integration. Layers above this will be excluded.

    Returns
    -------
    column_integrated_variable : xarray.DataArray
        A 3D array (time, lat, lon) of the vertically integrated variable
        in units adjusted by the factor (1 / 9.81), representing integration
        over pressure coordinates.

    Notes
    -----
    - The function constructs intermediate pressure levels at layer boundaries
      using the input `pressure_array`, and computes thicknesses and beta
      weighting factors accordingly.
    - The integration accounts for whether each layer is fully or partially
      beneath the surface using the surface pressure and beta factor.
    - Any layers with pressure less than `ptop` are excluded from the integration.

    References
    ----------
    Trenberth, K. E. (1991). Climate diagnostics from global analyses:
    Conservation of mass in ECMWF analyses. Journal of Climate, 4(7), 707–722.
    """

    # Convert the pressure array to be bottom-to-top, and to have index labels instead of level labels
    pressure_array = pressure_array.transpose(..., "lev")
    pressure_array = pressure_array.isel(lev=slice(None, None,-1))
    pressure_array = pressure_array.assign_coords(lev=("lev", np.arange(1, 2*len(pressure_array.lev), 2)))
    pressure_array = pressure_array.rename({'lev':'index'})

    fill_value = 0

    original_shape = list(pressure_array.shape)
    original_dims = list(pressure_array.dims)
    original_coords = pressure_array.coords

    # Modify shape: extend the last dimension by 1
    extended_shape = original_shape.copy()
    extended_shape[-1] += 1

    # Create extended coordinates for the last dimension
    last_dim = original_dims[-1]
    last_coord = original_coords[last_dim].values
    extended_last_coord = np.arange(0, 2*len(pressure_array.index)+2, 2)

    # Create a dict of coordinates, extending only the last dimension
    extended_coords = {
        dim: (original_coords[dim].values if dim != last_dim else extended_last_coord)
        for dim in original_dims
    }

    # Create the data with desired fill_value (e.g. zeros)
    intermediate_levels = xr.DataArray(
        data = np.full(extended_shape, fill_value, dtype=pressure_array.dtype),
        dims = pressure_array.dims,
        coords = extended_coords
    )

    # # Calculate the intermediate pressure levels
    # intermediate_levels = xr.DataArray(
    #     data=np.zeros((
    #         len(pressure_array.time),
    #         len(pressure_array.lat),
    #         len(pressure_array.lon),
    #         len(pressure_array.index)+1
    #     )),
    #     # dims=["time", "lat", "lon", "index"],
    #     dims=[*pressure_array.dims]
    #     coords={
    #         'time': pressure_array.time,
    #         'lat': pressure_array.lat,
    #         'lon': pressure_array.lon,
    #         'index': np.arange(0, 2*len(pressure_array.index)+2, 2)
    #     }
    # )

    intermediate_levels[..., 0] = pbot*np.ones_like(surface_pressure.values)
    intermediate_levels[..., 1:-1] = 0.5*(pressure_array[..., :-1].values + pressure_array[..., 1:].values)
    intermediate_levels[..., -1] = np.zeros_like(surface_pressure.values)

    # Calculate the thickness of each layer
    pressure_jminus1 = intermediate_levels[..., :-1].values
    pressure_jplus1 = intermediate_levels[..., 1:].values
    layer_thickness = xr.zeros_like(pressure_array)
    layer_thickness[..., :] = pressure_jminus1 - pressure_jplus1

    # Calculate the Trenberth beta factors
    reshaped_surface_pressure = surface_pressure.expand_dims(
        dim={"index": pressure_array.index},
        axis=surface_pressure.ndim
    )

    lower_one_condition = (pressure_jminus1 < reshaped_surface_pressure)
    lower_zero_condition = (pressure_jplus1 > reshaped_surface_pressure)
    lower_beta = xr.where(
        lower_one_condition, 1,
        xr.where(
            lower_zero_condition, 0, (
                (reshaped_surface_pressure - pressure_jplus1)
                / (pressure_jminus1 - pressure_jplus1)
            )
        )
    )

    upper_one_condition = (pressure_jplus1 > ptop)
    upper_zero_condition = (pressure_jminus1 < ptop)
    upper_beta = xr.where(
        upper_one_condition, 1,
        xr.where(
            upper_zero_condition, 0, (
                (pressure_jminus1 - ptop)
                / (pressure_jminus1 - pressure_jplus1)
            )
        )
    )

    # Re-order and label the variable to be integrated
    relabelled_variable = variable.isel(lev=slice(None, None, -1))
    relabelled_variable = relabelled_variable.assign_coords(lev=("lev", np.arange(1, 2*len(pressure_array.index), 2)))
    relabelled_variable = relabelled_variable.rename({'lev': 'index'})

    # Calculate the vertical integral
    column_integrated_variable = (1/9.8)*(
        lower_beta*relabelled_variable*layer_thickness*upper_beta
        # lower_beta*relabelled_variable*layer_thickness*
    ).sum(dim='index')
    return column_integrated_variable


def vertical_pressure_gradient(input_variable, input_pressure_array, units='hPa'):
    """
    Calculate the vertical pressure gradient of a variable using finite differences.

    Parameters
    ----------
    input_variable : xarray.DataArray
        The variable for which the vertical pressure gradient is to be calculated.
        Must include a vertical dimension named 'lev'.

    input_pressure_array : xarray.DataArray
        The pressure values corresponding to each vertical level. Must have the
        same shape and dimensions as `input_variable`, including the 'lev' dimension.

    Returns
    -------
    xarray.DataArray
        The vertical gradient of `input_variable` with respect to pressure,
        with the same dimensions and shape as the input.

    Notes
    -----
    - The gradient is computed using:
        - Forward difference at the bottom boundary (lev=0)
        - Central difference for interior levels (1 <= lev <= N-2)
        - Backward difference at the top boundary (lev=N-1)
    - Assumes pressure decreases with height (as in atmospheric applications).
    """

    # Reorient the input data so that the vertical level dimension 'lev' is last
    variable = input_variable.transpose(..., "lev")
    pressure_array = input_pressure_array.transpose(..., "lev")

    # Create an output array of zeros with the same shape as the input variable
    vertical_gradient = xr.zeros_like(variable)

    # Compute the forward difference for the bottom-most level (lev=0)
    vertical_gradient[..., 0] = (
        (variable[..., 1].values - variable[..., 0].values)
        / (pressure_array[..., 1].values - pressure_array[..., 0].values)
    )

    # Compute the central difference for interior levels (lev=1 to lev=-2)
    vertical_gradient[..., 1:-1] = (
        (variable[..., 2:].values - variable[..., :-2].values)
        / (pressure_array[..., 2:].values - pressure_array[..., :-2].values)
    )

    # Compute the backward difference for the top-most level (lev=-1)
    vertical_gradient[..., -1] = (
        (variable[..., -1].values - variable[..., -2].values)
        / (pressure_array[..., -1].values - pressure_array[..., -2].values)
    )

    # Revert to the original dimension ordering of the input variable
    output_vertical_gradient = vertical_gradient.transpose(*input_variable.dims)

    if units == 'hPa':
        output_vertical_gradient /= 100

    return output_vertical_gradient

In [18]:
save_column_integrated_data = True

# Initialize arrays for budget variables
column_horizontal_MSE_advection = {}
column_vertical_MSE_advection = {}
column_vertical_DSE_advection = {}

n_calculations = 3

variables_loaded = {}
for exp_index, experiment in enumerate(experiments_list):

    print(f"{'-'*str_width}")
    print(f"{f'Loading {experiment} data...':<{str_width}}")
    print(f"{'-'*str_width}")
    variable_data_files = sorted(glob.glob(
        rf"{data_directory}/{experiment}/daily_model-level_data/*.nc"
    ))
    for index, file in enumerate(variable_data_files):
        variable_data = xr.open_dataarray(file)
        print(f"{f'({index+1}/{len(variable_data_files)}) {variable_data.name}...':<{str_width-1}}", end="")
        variables_loaded[variable_data.name] = variable_data.sel(time=slice(START_TIME, END_TIME))
        print(rf"{'✔':>1}")
    print(f"{'-'*str_width}")

    recalculate_budgets = True

    non_timed_data =  xr.open_dataset(
        rf"/glade/campaign/univ/uwas0114/SST_AQP3_Qobs_27_-4K_3h_10y/atm/hist/SST_AQP3_Qobs_27_-4K_3h_20y_new2.cam.h1.0001-02-16-43200.nc"
    )

    if recalculate_budgets:
        print(f"{'Pressure Array...':<{str_width-1}}", end="")
        lower_level_pressure = 1100.*100.
        # upper_level_pressure = 100.*100.
        surface_pressure = variables_loaded['PS']
        pressure_array = non_timed_data['hyam']*non_timed_data['P0'] + non_timed_data['hybm']*surface_pressure
        pressure_array = pressure_array.transpose("time", "lev", "lat", "lon")
        print(rf"{'✔':>1}")

        time_mean_temperature = variables_loaded['T'].mean(dim=['time'])
        temperature_minimum_index = time_mean_temperature.argmin('lev')
        temperature_minimum_pressure_level = pressure_array.mean(dim='time').transpose("lev", ...)[temperature_minimum_index].mean(dim=['lat', 'lon'])
        upper_level_pressure = temperature_minimum_pressure_level.values

        print(f"{'Moist Static Energy...':<{str_width-1}}", end="")
        GRAVITY = 9.8                        #  m/s^2
        SPECIFIC_HEAT = 1005.7               #  J/Kg*K ; specific heat at constant pressure for dry air
        HEAT_OF_VAPORIZATION = 2.501e6       #  [J/kg]=[m2/s2]  Latent Heat of Vaporization at 0
        HEAT_OF_FUSION = 3.337e5             # [J/kg]=[m2/s2]  Latent Heat of Sublimation at 0
        dry_static_energy = (
            + SPECIFIC_HEAT*variables_loaded['T']
            + GRAVITY*variables_loaded['Z3']
        )
        dry_static_energy.name = 'Dry Static Energy'
        dry_static_energy.attrs['units'] = r"J kg$^{-2}$"

        moist_static_energy = (
            HEAT_OF_VAPORIZATION*variables_loaded['Q']
            - HEAT_OF_FUSION*variables_loaded['CLDICE']
            + SPECIFIC_HEAT*variables_loaded['T']
            + GRAVITY*variables_loaded['Z3']
        )
        moist_static_energy.name = 'Moist Static Energy'
        moist_static_energy.attrs['units'] = r"J kg$^{-2}$"
        print(rf"{'✔':>1}")


        print(f"{'Zonal Advection...':<{str_width-1}}", end="")
        zonal_MSE_gradient = (
            (180/np.pi)
            * moist_static_energy.differentiate('lon')
            / (EARTH_RADIUS*np.cos(np.deg2rad(moist_static_energy.lat)))
        )
        zonal_advection = variables_loaded['U']*zonal_MSE_gradient
        zonal_advection.name = 'Zonal Advection'
        zonal_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"
        print(rf"{'✔':>1}")

        print(f"{'Meridional Advection...':<{str_width-1}}", end="")
        meridional_MSE_gradient = (
            (180/np.pi)
            * moist_static_energy.differentiate('lat')
            / EARTH_RADIUS
        )

        meridional_advection = variables_loaded['V'] * meridional_MSE_gradient
        meridional_advection.name = 'Meridional Advection'
        meridional_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"
        print(rf"{'✔':>1}")

        horizontal_MSE_advection = zonal_advection + meridional_advection
        horizontal_MSE_advection.name = 'Horizontal Advection'
        horizontal_MSE_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"

        print(f"{'Vertical MSE Advection...':<{str_width-1}}", end="")
        # vertical_MSE_gradient = (1/100)*moist_static_energy.differentiate('lev')
        vertical_MSE_gradient = xr.zeros_like(moist_static_energy)
        vertical_MSE_gradient[:, 0] = (
            (moist_static_energy[:, 1].values - moist_static_energy[:, 0].values)
            / (pressure_array[:, 1].values - pressure_array[:, 0].values)
        )
        vertical_MSE_gradient[:, 1:-1] = (
            (moist_static_energy[:, 2:].values - moist_static_energy[:, :-2].values)
            / (pressure_array[:, 2:].values - pressure_array[:, :-2].values)
        )
        vertical_MSE_gradient[:, -1] = (
            (moist_static_energy[:, -1].values - moist_static_energy[:, -2].values)
            / (pressure_array[:, -1].values - pressure_array[:, -2].values)
        )

        vertical_MSE_advection = variables_loaded['OMEGA'] * vertical_MSE_gradient
        vertical_MSE_advection.name = 'Vertical MSE Advection'
        vertical_MSE_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"
        print(rf"{'✔':>1}")

        print(f"{'Vertical DSE Advection...':<{str_width-1}}", end="")
        # vertical_MSE_gradient = (1/100)*moist_static_energy.differentiate('lev')
        vertical_DSE_gradient = xr.zeros_like(dry_static_energy)
        vertical_DSE_gradient[:, 0] = (
            (dry_static_energy[:, 1].values - dry_static_energy[:, 0].values)
            / (pressure_array[:, 1].values - pressure_array[:, 0].values)
        )
        vertical_DSE_gradient[:, 1:-1] = (
            (dry_static_energy[:, 2:].values - dry_static_energy[:, :-2].values)
            / (pressure_array[:, 2:].values - pressure_array[:, :-2].values)
        )
        vertical_DSE_gradient[:, -1] = (
            (dry_static_energy[:, -1].values - dry_static_energy[:, -2].values)
            / (pressure_array[:, -1].values - pressure_array[:, -2].values)
        )

        vertical_DSE_advection = variables_loaded['OMEGA'] * vertical_DSE_gradient
        vertical_DSE_advection.name = 'Vertical DSE Advection'
        vertical_DSE_advection.attrs['units'] = r"J kg$^{-2}$ s$^{-1}$"
        print(rf"{'✔':>1}")

        print(f"{'Column integrating...':<{str_width-1}}", end="")
        # Horizontal MSE Advection
        column_horizontal_MSE_advection[experiment] = beta_column_integrate(
            pressure_array,
            horizontal_MSE_advection,
            surface_pressure,
            lower_level_pressure,
            upper_level_pressure
        )
        column_horizontal_MSE_advection[experiment].name = 'Horizontal MSE Advection'
        column_horizontal_MSE_advection[experiment].attrs = {}
        column_horizontal_MSE_advection[experiment].attrs['long_name'] = "Horizontal of Moist Static Energy"
        column_horizontal_MSE_advection[experiment].attrs['math_name'] = r"$\langle$$\vec{v} \nabla$h$\rangle$"
        column_horizontal_MSE_advection[experiment].attrs['units'] = r"W m$^{-2}$"

        # Vertical MSE Advection
        column_vertical_MSE_advection[experiment] = beta_column_integrate(
            pressure_array,
            vertical_MSE_advection,
            surface_pressure,
            lower_level_pressure,
            upper_level_pressure
        )
        column_vertical_MSE_advection[experiment].name = 'Vertical MSE Advection'
        column_vertical_MSE_advection[experiment].attrs = {}
        column_vertical_MSE_advection[experiment].attrs['long_name'] = "Vertical Advection of Moist Static Energy"
        column_vertical_MSE_advection[experiment].attrs['math_name'] = r"$-\langle$$ω \partial_{p}$h$\rangle$"
        column_vertical_MSE_advection[experiment].attrs['units'] = r"W m$^{-2}$"

        # Vertical MSE Advection
        column_vertical_DSE_advection[experiment] = beta_column_integrate(
            pressure_array,
            vertical_DSE_advection,
            surface_pressure,
            lower_level_pressure,
            upper_level_pressure
        )
        column_vertical_DSE_advection[experiment].name = 'Vertical DSE Advection'
        column_vertical_DSE_advection[experiment].attrs = {}
        column_vertical_DSE_advection[experiment].attrs['long_name'] = "Vertical Advection of Dry Static Energy"
        column_vertical_DSE_advection[experiment].attrs['math_name'] = r"$-\langle$$ω \partial_{p}$d$\rangle$"
        column_vertical_DSE_advection[experiment].attrs['units'] = r"W m$^{-2}$"
        print(rf"{'✔':>1}")

GMS_variables = {
    'Horizontal MSE Advection': column_horizontal_MSE_advection,
    'Vertical MSE Advection': column_vertical_MSE_advection,
    'Vertical DSE Advection': column_vertical_DSE_advection,
}

print("Concatenating along experiment axis...")
print(f"{'-'*str_width}")
multi_experiment_GMS_variables = {}
for index, GMS_variable in enumerate(GMS_variables):
    print(f"{f'({index+1}/{len(GMS_variables)}) {GMS_variable}...':<{str_width-1}}", end="")
    multi_experiment_GMS_variables[GMS_variable] = xr.concat(
        [GMS_variables[GMS_variable][experiment] for experiment in experiments_list],
        dim=experiments_list
    )
    multi_experiment_GMS_variables[GMS_variable] = multi_experiment_GMS_variables[GMS_variable].rename(
        {"concat_dim": "experiment"}
    )
    print(rf"{'✔':>1}")

if save_column_integrated_data:
    print(f"{'Saving budget terms':^{str_width}}")
    print(f"{'='*str_width}")

    for index, (variable_name, variable_data) in enumerate(multi_experiment_GMS_variables.items()):
        print(f"{f'({index+1}/{len(multi_experiment_GMS_variables)}) {variable_name}...':<{str_width-1}}", end="")

        filename = f"{data_directory}/MSE_budget_terms/daily_model-level_data/multi_experiment_{variable_name.lower().replace(' ', '_')}.nc"
        if os.path.exists(filename):
            # Prompt user for confirmation
            # response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
            response = 'y'
            if response == 'y':
                os.remove(filename)  # Delete the existing file
                variable_data.to_netcdf(filename)  # Save the new file
                print(rf"{'✔ (overwritten)':>1}")
            else:
                print(rf"{'✘ (skipped)':>1}")
        else:
            variable_data.to_netcdf(filename)  # Save the new file
            print(rf"{'✔':>1}")
else:
    print(f"{'Not saving budget terms':<{str_width}}")

print(f"{'='*str_width}")
print("Finished")

----------------------------------------
Loading -4K data...                     
----------------------------------------
(1/17) CLDICE...                       ✔
(2/17) FLNS...                         ✔
(3/17) FLNT...                         ✔
(4/17) FLUT...                         ✔
(5/17) FSNS...                         ✔
(6/17) FSNT...                         ✔
(7/17) LHFLX...                        ✔
(8/17) OMEGA...                        ✔
(9/17) PRECC...                        ✔
(10/17) PRECL...                       ✔
(11/17) PS...                          ✔
(12/17) Q...                           ✔
(13/17) SHFLX...                       ✔
(14/17) T...                           ✔
(15/17) U...                           ✔
(16/17) V...                           ✔
(17/17) Z3...                          ✔
----------------------------------------
Pressure Array...                      ✔
Moist Static Energy...                 ✔
Zonal Advection...                     ✔
Meridional Advec